In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

sys.path.append('../..//')
from utils_mitgcm import open_mitgcm_ds_from_config
import pylake

In [ ]:
model = 'neuchatel_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('../../config.json', model)

In [ ]:
folder_path = os.path.dirname(mitgcm_config['datapath'])
output_folder = os.path.join(folder_path, "seiche_analysis", "potential_energy")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
grid_resolution = 100
ds['YC'] = np.arange(1, len(ds['YC']) + 1) * grid_resolution - grid_resolution / 2
ds['XC'] = np.arange(1, len(ds['XC']) + 1) * grid_resolution - grid_resolution / 2
ds['YG'] = np.arange(0, len(ds['YG'])) * grid_resolution
ds['XG'] = np.arange(0, len(ds['XG'])) * grid_resolution

# Compute Potential energy

In [ ]:
g = 9.81
rho0 = 1000.0  # reference density kg/m3

In [ ]:
# ---------------------------
# 1. Convert temperature to density
# ---------------------------
rho = pylake.dens0(s=0.2, t=ds['THETA']).where(ds['THETA']>0)

# ---------------------------
# 2. Compute time-mean density profile
# ---------------------------
rho_mean = rho.mean(dim=['XC', 'YC'], skipna=True).rolling(time=72, center=True).mean()

In [ ]:
# ---------------------------
# 3. Vertical density gradient
# ---------------------------
drho_dz = rho_mean.differentiate("Z")

In [ ]:
# ---------------------------
# 4. Compute vertical displacement η
# Task: for each (time, XC, YC, Z), find the depth Z0 where rho_mean(time, Z0) is closest to rho(time, Z, XC, YC),
#       then compute η = Z - Z0
# ---------------------------

rho_prime = rho - rho_mean  # kept for continuity (not used below)

z = rho["Z"]


def _nearest_z0_from_rho(rho_vals_1d, rho_mean_1d, z_1d):
    rho_vals_1d = np.asarray(rho_vals_1d)
    rho_mean_1d = np.asarray(rho_mean_1d)
    z_1d = np.asarray(z_1d)

    out = np.full_like(rho_vals_1d, np.nan, dtype=float)

    m = np.isfinite(rho_mean_1d) & np.isfinite(z_1d)
    if np.count_nonzero(m) < 2:
        return out

    rm = rho_mean_1d[m].astype(float)
    zm = z_1d[m].astype(float)

    # sort by mean density to enable fast nearest lookup
    order = np.argsort(rm)
    rm = rm[order]
    zm = zm[order]

    v = rho_vals_1d.astype(float)
    mv = np.isfinite(v)
    if not np.any(mv):
        return out

    vv = v[mv]
    idx = np.searchsorted(rm, vv, side="left")

    idx0 = np.clip(idx - 1, 0, rm.size - 1)
    idx1 = np.clip(idx, 0, rm.size - 1)

    d0 = np.abs(vv - rm[idx0])
    d1 = np.abs(vv - rm[idx1])
    choose1 = d1 < d0
    best_idx = np.where(choose1, idx1, idx0)

    # mask out-of-range values (beyond min/max mean density)
    in_range = (vv >= rm[0]) & (vv <= rm[-1])
    z0 = np.full_like(vv, np.nan, dtype=float)
    z0[in_range] = zm[best_idx[in_range]]

    out[mv] = z0
    return out


z0 = xr.apply_ufunc(
    _nearest_z0_from_rho,
    rho,
    rho_mean,
    z,
    input_core_dims=[["Z"], ["Z"], ["Z"]],
    output_core_dims=[["Z"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)

eta = (z - z0).where(np.isfinite(z0))


In [ ]:
# ---------------------------
# 5. Buoyancy frequency
# ---------------------------

N2 = -g / rho0 * np.abs(drho_dz)

# ---------------------------
# 6. Potential energy density profile
# ---------------------------

Ep_profile = 0.5 * rho0 * N2 * (eta**2)

# ---------------------------
# 7. Integrate vertically
# ---------------------------

E = Ep_profile.fillna(0).integrate("Z").where(ds['THETA'].isel(Z=0)>0)
# units: J/m2

In [ ]:
# ---------------------------
# 8. Energy anomaly
# ---------------------------

E_anomaly = (Ep_profile-Ep_profile.mean(dim=['XC','YC'])).fillna(0).integrate("Z").where(ds['THETA'].isel(Z=0)>0)

In [ ]:
t_idx=80

plt.close('all')
plt.figure(figsize=(16,5))

E.isel(time=t_idx).plot(cmap='jet', vmin=-200)

plt.text(0.02, 0.98, f'{np.datetime_as_string(ds.time.isel(time=t_idx), unit="s").replace("T", " ")}', transform=plt.gca().transAxes, ha='left', va='top')
plt.title('')
plt.tight_layout()

In [ ]:
E_pot = E.fillna(0).integrate(['XC', 'YC'])

In [ ]:
E_pot.isel(time=range(72,len(ds.time))).plot()

In [ ]:
df_epot_tot = E_pot.isel(time=range(72,len(ds.time))).to_dataframe(name='epot_j_total')['epot_j_total']

In [ ]:
df_epot_tot.reset_index().to_csv(os.path.join(output_folder, "epot_whole_lake.csv"))

for t_idx in range(72, len(ds.time)-1):
    plt.close('all')
    plt.figure(figsize=(16,5))

    E.isel(time=t_idx).plot(cmap='jet', vmax=500)

    plt.text(0.02, 0.98, f'{np.datetime_as_string(ds.time.isel(time=t_idx), unit="s").replace("T", " ")}', transform=plt.gca().transAxes, ha='left', va='top')
    plt.title('')
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'Epot_t{t_idx}.png'))